# MGGP — Forrester 1D & Koza-1
**Multi-population symbolic regression** using `SymGeneEvolver` with two simultaneous populations on 1D analytic benchmark functions.

Figures produced:
- `fig01_mggp_obs_pred.png` — Observed × Predicted (train / val / test)
- `fig01_mggp_fitness_decay.png` — Fitness decay over generations
- `fig01_mggp_curves.png` — True function vs MGGP prediction curve


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from sklearn.metrics import r2_score
import os, math

FIG_DIR = Path("figures")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_COLOR = '#1f77b4'
VAL_COLOR   = '#ff7f0e'
TEST_COLOR  = '#2ca02c'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.linestyle':   '-',
    'grid.alpha':       0.4,
    'grid.color':       '#cccccc',
    'font.size':        11,
    'axes.labelsize':   12,
    'axes.titlesize':   13,
    'legend.fontsize':  10,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

def obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title, xlabel="Observed", ylabel="Predicted"):
    h1 = ax.scatter(y_tr, yp_tr, c=TRAIN_COLOR, marker='o', s=40, alpha=0.75,
                    label=f'Train  (R²={r2_score(y_tr, yp_tr):.6f})')
    h2 = ax.scatter(y_v,  yp_v,  c=VAL_COLOR,   marker='s', s=40, alpha=0.75,
                    label=f'Val    (R²={r2_score(y_v,  yp_v):.6f})')
    h3 = ax.scatter(y_te, yp_te, c=TEST_COLOR,  marker='^', s=40, alpha=0.75,
                    label=f'Test   (R²={r2_score(y_te, yp_te):.6f})')
    all_y = np.concatenate([y_tr, y_v, y_te])
    lo, hi = all_y.min(), all_y.max()
    ax.plot([lo, hi], [lo, hi], color='black', lw=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    return h1, h2, h3

print("Style loaded.")

from symgene.callbacks import Callback

class HistoryRecorder(Callback):
    def __init__(self):
        self.records = []
    def on_generation_end(self, gen, logs=None):
        if logs:
            self.records.append(dict(logs))
        return None

def _snap(ax, log_y=False):
    """Tick marks on all 4 sides; snap x to outermost ticks, y to full decades (log) or ticks (linear)."""
    ax.figure.canvas.draw()
    ax.tick_params(top=True, right=True, which='both', direction='in')

    lo, hi = ax.get_xlim()
    xt = sorted(t for t in ax.get_xticks() if lo - 1e-9 <= t <= hi + 1e-9)
    if len(xt) >= 2:
        ax.set_xlim(xt[0], xt[-1])

    if log_y:
        lo, hi = ax.get_ylim()
        if lo > 0 and hi > 0:
            lo_dec = 10 ** math.floor(math.log10(lo))
            hi_log = math.log10(hi)
            frac   = hi_log - math.floor(hi_log)
            hi_dec = 10 ** (math.floor(hi_log) if frac < 0.02 else math.ceil(hi_log))
            if lo_dec < hi_dec:
                ax.set_ylim(lo_dec, hi_dec)
    else:
        lo, hi = ax.get_ylim()
        yt = sorted(t for t in ax.get_yticks() if lo - 1e-9 <= t <= hi + 1e-9)
        if len(yt) >= 2:
            ax.set_ylim(yt[0], yt[-1])

def _fix_cbar(cb, cp):
    """Add tick labels at the very top and bottom of a colorbar."""
    _lo, _hi = cp.get_clim()
    _t = [x for x in cb.get_ticks() if _lo < x < _hi]
    cb.set_ticks([_lo] + _t + [_hi])


## Data generation
70 / 15 / 15 split. Functions known analytically → generate as many points as needed.

In [ ]:

from symgene.benchmarks import forrester_1d

bench = forrester_1d()
koza_fn = lambda x: x**4 + x**3 + x**2 + x

rng = np.random.default_rng(0)
# Random (unsorted) split so train/val/test all cover the full domain [0,1]
X_all = rng.uniform(0, 1, (200, 1))
idx   = rng.permutation(200)
y_f_all = np.array([bench.fn(X_all[i]) for i in range(200)])
y_k_all = koza_fn(X_all[:, 0])

n_train, n_val = 140, 30           # 70 / 15 / 15 split
i_tr, i_v, i_te = idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

X_train, X_val, X_test = X_all[i_tr], X_all[i_v], X_all[i_te]
y_f_train, y_f_val, y_f_test = y_f_all[i_tr], y_f_all[i_v], y_f_all[i_te]
y_k_train, y_k_val, y_k_test = y_k_all[i_tr], y_k_all[i_v], y_k_all[i_te]

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


## Model setup and training

In [ ]:
from symgene import PrimitiveSet, Population, SymGeneEvolver
from symgene.primitives import STANDARD
from symgene.fitness import FitnessEvaluator
from symgene.metrics.regression import mse
from symgene.metrics.complexity import complexity_penalty
from symgene.metrics import rmse, nrmse
from symgene.selection import TournamentSelection
from symgene.callbacks import EarlyStopping, GenerationLogger

history_cb = HistoryRecorder()

pset = PrimitiveSet(n_inputs=1, feature_names=["x"])
pset.add_from_catalog(STANDARD)
pset.set_squash(lim=8, alpha=0.1, scale=2.0)
pset.add_custom(fn=lambda x: float(np.exp(-((x - 0.5)**2) / 0.1)), arity=1, name="bump")

pop_forrester = Population(
    name="forrester", pset=pset,
    n_genes=1, n_genes_max=10, pop_size=50,  # start with 1 gene — grows via add-mutation
    elite_ratio=0.025,
    tree_min=2, tree_max=40,
    tree_init_max=2,   # shallow init → poor initial approximation → visible decay
    height_max=8,
    cxpb=0.90, cxpb_low=0.40,
    mutpb=0.35, mutpb_low=0.25,              # more mutations to drive gene addition
    mutation_weights=[0.2, 2.0, 0.8],        # strongly favor "add gene"
    fitness=FitnessEvaluator(metric=mse, penalties=[complexity_penalty(lambda_=1e-4)]),
    selection=TournamentSelection(size=2),
)
pop_koza = Population(
    name="koza1", pset=pset,
    n_genes=1, n_genes_max=8, pop_size=50,
    elite_ratio=0.025,
    tree_min=2, tree_max=30,
    tree_init_max=2,
    height_max=8,
    cxpb=0.90, cxpb_low=0.50,
    mutpb=0.35, mutpb_low=0.25,
    mutation_weights=[0.2, 2.0, 0.8],
    fitness=FitnessEvaluator(metric=mse, penalties=[complexity_penalty(lambda_=5e-4)]),
    selection=TournamentSelection(size=5),
)

evolver = SymGeneEvolver(
    populations=[pop_forrester, pop_koza], n_gen=300,
    cross_population=False, seed=0,
    callbacks=[history_cb, GenerationLogger(every=20)],
    verbose=0,
)
results = evolver.fit(
    X_train,
    {"forrester": y_f_train, "koza1": y_k_train},
    X_val=X_val,
    y_val={"forrester": y_f_val, "koza1": y_k_val},
)
print("Training complete.")


## Figure 1 — Observed × Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plt.subplots_adjust(wspace=0.35, bottom=0.12)

configs = [
    (
        "forrester",
        "Forrester  —  (6x−2)²·sin(12x−4)",
        y_f_train,
        y_f_val,
        y_f_test,
    ),
    ("koza1", "Koza-1  —  x⁴+x³+x²+x", y_k_train, y_k_val, y_k_test),
]

for ax, (pop_name, title, y_tr, y_v, y_te) in zip(axes, configs):
    res = results[pop_name]
    yp_tr = res.predict(X_train)
    yp_v = res.predict(X_val)
    yp_te = res.predict(X_test)

    # Draw base figure
    obs_pred_ax(
        ax,
        y_tr,
        yp_tr,
        y_v,
        yp_v,
        y_te,
        yp_te,
        title,
        xlabel="Observed",
        ylabel="Predicted",
    )

    # --- Remove borders/lines from plotted points ---
    # 1. For points generated via ax.scatter (PathCollections)
    for collection in ax.collections:
        collection.set_linewidth(0)  # Remove circle border line
        collection.set_edgecolor("none")  # Ensure no visible border

    # 2. For points generated via ax.plot (Lines2D with markers)
    for line in ax.lines:
        # Keep only the diagonal/reference line; strip continuous lines through markers
        if line.get_marker() != "None" and line.get_marker() != "":
            line.get_marker()
            line.set_linewidth(0)  # Remove line between/through points
            line.set_markeredgewidth(0)  # Remove marker border

    # 3. Apply _snap
    _snap(ax)

    # 4. Add grid
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

    # 5. Black borders and axes
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # 6. Clean legend
    leg = ax.legend(
        loc="upper left",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
        handlelength=1.2,
        handletextpad=0.5,
    )

    for handle in leg.legend_handles:
        if hasattr(handle, "set_linewidth"):
            handle.set_linewidth(0)

plt.savefig(os.path.join(FIG_DIR, "fig01_mggp_obs_pred.png"), dpi=300)
plt.show()
print("Saved: fig01_mggp_obs_pred.png")

## Figure 2 — Fitness decay over generations

In [ ]:
import pandas as pd

df = pd.DataFrame(history_cb.records)
print("Available columns:", df.columns.tolist())
print(f"Generations recorded: {len(df)}")

pop_configs = [
    ("forrester", "Forrester  —  (6x-2)²·sin(12x-4)"),
    ("koza1", "Koza-1  —  x⁴+x³+x²+x"),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plt.subplots_adjust(wspace=0.45, bottom=0.15)

for ax, (pop_name, func_label) in zip(axes, pop_configs):
    col_tr = f"{pop_name}_train_mse"

    if col_tr in df.columns:
        ax.semilogy(
            df["gen"],
            df[col_tr],
            color=TRAIN_COLOR,
            lw=2.5,
            zorder=5,
            label="Best individual",
        )

    ax.set_xlabel("Generation", color="black")
    ax.set_ylabel("Fitness — penalised MSE (log scale)", color="black")
    ax.set_title(func_label, color="black")
    ax.legend(
        loc="upper right", frameon=True, edgecolor="black", fancybox=False
    )

    # 1. Apply _snap styling first
    _snap(ax, log_y=True)

    # 2. Add grid (major and minor lines for log scale)
    ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

    # 3. Ensure visible black borders and tick marks
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

plt.savefig(os.path.join(FIG_DIR, "fig01_mggp_fitness_decay.png"), dpi=300)
plt.show()
print("Saved: fig01_mggp_fitness_decay.png")

## Figure 3 — True function vs MGGP prediction

In [ ]:
x_grid = np.linspace(0, 1, 400).reshape(-1, 1)
_idx = np.linspace(0, 399, 50, dtype=int)

configs = [
    (
        "forrester",
        "Forrester  —  (6x−2)²·sin(12x−4)",
        lambda x: (6 * x - 2) ** 2 * np.sin(12 * x - 4),
    ),
    ("koza1", "Koza-1  —  x⁴+x³+x²+x", lambda x: x**4 + x**3 + x**2 + x),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plt.subplots_adjust(wspace=0.35, bottom=0.15)

for ax, (pop_name, title, true_fn) in zip(axes, configs):
    y_true = true_fn(x_grid[:, 0])
    y_mggp = results[pop_name].predict(x_grid)

    ax.plot(
        x_grid[:, 0], y_true, color="black", lw=2.0, label="True function"
    )
    ax.scatter(
        x_grid[_idx, 0],
        y_mggp[_idx],
        color="red",
        s=8,
        alpha=0.85,
        zorder=5,
        label="MGGP prediction",
    )

    ax.set_xlabel("x", color="black")
    ax.set_ylabel("f(x)", color="black")
    ax.set_title(title, color="black")
    ax.legend(loc="best", frameon=True, edgecolor="black", fancybox=False)

    # 1. Apply _snap first (may reset style/axes)
    if "_snap" in globals():
        _snap(ax)

    # 2. Set limits after _snap to force correct display
    if pop_name == "forrester":
        # True Forrester minimum ≈ -6.02, maximum ≈ 15.83
        ax.set_ylim(-7.5, 17.5)

    # Ensure borders and labels remain black
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

plt.savefig(os.path.join(FIG_DIR, "fig01_mggp_curves.png"), dpi=300)
plt.show()
print("Saved: fig01_mggp_curves.png")

## Numerical results

In [19]:

from symgene.metrics import rmse, nrmse

for pop_name, y_tr, y_v, y_te in [
        ("forrester", y_f_train, y_f_val, y_f_test),
        ("koza1",     y_k_train, y_k_val, y_k_test)]:
    res = results[pop_name]
    yp_tr = res.predict(X_train)
    yp_v  = res.predict(X_val)
    yp_te = res.predict(X_test)
    print(f"\n{'='*55}")
    print(f"Population : {pop_name.upper()}")
    print(f"Genes      : {res.n_genes_}")
    print(f"Expression : {res.best_expression_}")
    print(f"Train  RMSE={rmse(y_tr, yp_tr):.4f}  NRMSE={nrmse(y_tr, yp_tr):.4f}  R²={r2_score(y_tr, yp_tr):.4f}")
    print(f"Val    RMSE={rmse(y_v,  yp_v):.4f}  NRMSE={nrmse(y_v, yp_v):.4f}  R²={r2_score(y_v,  yp_v):.4f}")
    print(f"Test   RMSE={rmse(y_te, yp_te):.4f}  NRMSE={nrmse(y_te, yp_te):.4f}  R²={r2_score(y_te, yp_te):.4f}")



Population : FORRESTER
Genes      : 10
Expression : gaussian(bump(abs(min2(cube(cube(mean3(x, x, x))), mul(sigmoid(relu(x)), sin(exp(x))))))) | cube(add(x, cube(max3(sqrt(x), x, x)))) | gaussian(bump(abs(cube(max3(x, x, log(relu(x))))))) | sin(cos(x)) | sub(gaussian(add(sigmoid(relu(x)), inv(x))), inv(x)) | log(square(x)) | log(bump(abs(cube(max3(x, x, mul(max3(div(x, x), log(x), cube(x)), min2(cos(mul(div(x, x), gaussian(max3(x, x, mul(x, x))))), sigmoid(x)))))))) | gaussian(bump(abs(cube(max3(x, x, log(x)))))) | log(exp(mean3(square(add(bump(abs(cube(x))), cube(max3(sqrt(x), x, x)))), square(x), inv(mean3(x, x, x))))) | bump(square(add(bump(abs(cube(x))), x)))
Train  RMSE=0.0982  NRMSE=0.0045  R²=0.9996
Val    RMSE=0.1157  NRMSE=0.0053  R²=0.9994
Test   RMSE=0.1050  NRMSE=0.0051  R²=0.9995

Population : KOZA1
Genes      : 8
Expression : bump(x) | square(exp(x)) | cube(mean2(exp(x), x)) | relu(x) | bump(x) | square(exp(x)) | relu(x) | square(x)
Train  RMSE=0.0014  NRMSE=0.0004  R²=1.